In [0]:
# =====================================================================
# proceso / 01_ingest_superstore.py
# Task "ingest_superstore" (Bronze - Extract). PySpark puro.
# La tabla bronze.superstore_raw ya existe (creada en PrepAmb con DDL
# explícito), por lo que se usa insertInto() en vez de saveAsTable(),
# igual al patrón visto en clase. El orden de columnas del select debe
# coincidir EXACTAMENTE con el orden de columnas del CREATE TABLE.
# =====================================================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("raw_path", "abfss://raw@adlsretailproject0826.dfs.core.windows.net/superstore/")
dbutils.widgets.text("catalogo", "retail_medallion")
raw_path = dbutils.widgets.get("raw_path")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
# Columnas verificadas contra el CSV real (mismo orden que abajo, aunque
# el header trae espacios/guiones: "Row ID", "Sub-Category", etc. - no
# es problema: con schema explicito + header=True, Spark aplica el
# schema POR POSICION e ignora el texto exacto del header).
superstore_schema = StructType([
    StructField("Row_ID", IntegerType(), True),
    StructField("Order_ID", StringType(), True),
    StructField("Order_Date", StringType(), True),
    StructField("Ship_Date", StringType(), True),
    StructField("Ship_Mode", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Name", StringType(), True),
    StructField("Segment", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Postal_Code", StringType(), True),  # STRING a proposito: se pierden ceros a la izquierda en la fuente (ej. "1852"), no se puede recuperar aqui
    StructField("Region", StringType(), True),
    StructField("Product_ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Sub_Category", StringType(), True),
    StructField("Product_Name", StringType(), True),
    StructField("Sales", DoubleType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Profit", DoubleType(), True),
])

In [0]:
# El CSV real NO esta en UTF-8 (falla con byte invalido) - es Latin-1 /
# Windows-1252, comun en este dataset. Sin esta opcion, Spark leeria mal
# cualquier caracter especial en nombres de ciudad/producto.
df_raw = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .option("encoding", "ISO-8859-1")
    .schema(superstore_schema)
    .csv(raw_path)
)

df_bronze = (
    df_raw
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source_system", F.lit("superstore_kaggle"))
    .select(  # mismo orden que bronze.superstore_raw en PrepAmb
        "Row_ID", "Order_ID", "Order_Date", "Ship_Date", "Ship_Mode",
        "Customer_ID", "Customer_Name", "Segment", "Country", "City",
        "State", "Postal_Code", "Region", "Product_ID", "Category",
        "Sub_Category", "Product_Name", "Sales", "Quantity", "Discount",
        "Profit", "_source_file", "_ingest_timestamp", "_source_system",
    )
)

In [0]:
df_bronze.write.mode("overwrite").insertInto(f"{catalogo}.bronze.superstore_raw")

print(f"Bronze OK -> {catalogo}.bronze.superstore_raw ({df_bronze.count()} filas)")

Bronze OK -> retail_medallion.bronze.superstore_raw (9994 filas)
